# Per-cell features for a model

`cell_stats()` turns detections into statistics on any grid: fire-days per
km², the share of cropland under the fires, the distance from the nearest
static source and how far to trust the cell. Limen uses it on its own 1 km
grid; here the 1 km ETRS89-LAEA grid of the library (`LaeaGrid`).

A cell with nothing read stays **not measured** (`None`, NaN in a
dataframe), never 0: a model must not learn that the unknown is calm.

Offline, on the crop of the Palermo fires of July 2023 in the benchmark
(rows and a WorldCover window): no key, no download.

In [1]:
import csv
import io
from pathlib import Path

from firms_client import LaeaGrid, WorldCover, cell_stats, parse_csv, select

CASE = Path.cwd().parent / "benchmarks" / "data" / "palermo-2023"
rows = parse_csv((CASE / "rows.csv").read_text(encoding="utf-8"))
land = WorldCover(str(CASE / "worldcover"))
grid = LaeaGrid()
print(f"{len(rows)} detections, {len(select(rows))} vegetation fires")

11 detections, 11 vegetation fires


## One row per cell

The cells with fires, and two empty cells asked on purpose (`cells=`): an
empty cell is measured, with 0 fire-days, but has no land under a fire to
describe, so `cropland` stays `None`.

In [2]:
burnt = {grid.cell_of(h.latitude, h.longitude) for h in select(rows)}
x, y = next(iter(sorted(burnt)))
quiet = [(x + 20, y + 20), (x - 20, y - 20)]
stats = cell_stats(rows, grid, cells=quiet, land=land)
print(f"{'cell':>14} {'fire-days':>9} {'per km2':>8} {'cropland':>9} {'natural':>8}")
for c, s in sorted(stats.items(), key=lambda kv: -(kv[1].fire_days or 0))[:12]:
    crop = "None" if s.cropland is None else f"{s.cropland:.2f}"
    nat = "None" if s.natural is None else f"{s.natural:.2f}"
    print(f"{c!s:>14} {s.fire_days:>9} {s.fire_days_per_km2:>8.2f} {crop:>9} {nat:>8}")

          cell fire-days  per km2  cropland  natural
  (1667, 4614)         1     1.00      0.00     1.00
  (1667, 4613)         1     1.00      0.00     1.00
  (1668, 4615)         1     1.00      0.00     1.00
  (1668, 4614)         1     1.00      0.00     0.99
  (1669, 4615)         1     1.00      0.00     1.00
  (1669, 4614)         1     1.00      0.00     1.00
  (1669, 4613)         1     1.00      0.00     0.97
  (1670, 4614)         1     1.00      0.00     0.96
  (1687, 4633)         0     0.00      None     None
  (1647, 4593)         0     0.00      None     None


## Into a table for a model

`as_dict()` keeps `None`. Written to CSV it is an empty field, which pandas
reads as NaN: never fill it with 0.

In [3]:
out = io.StringIO()
fields = list(next(iter(stats.values())).as_dict())
writer = csv.DictWriter(out, fieldnames=fields)
writer.writeheader()
for s in stats.values():
    writer.writerow(s.as_dict())
print(out.getvalue()[:600])

cell,area_km2,measured,detections,fire_days,fire_days_per_km2,cropland,natural,land_days,static_km,score,reason
"(1667, 4614)",1.0,True,1,1,1.0,0.0,1.0,1,,1.0,
"(1667, 4613)",1.0,True,1,1,1.0,0.0,1.0,1,,1.0,
"(1668, 4615)",1.0,True,1,1,1.0,0.0,0.9999999999999999,1,,1.0,
"(1668, 4614)",1.0,True,1,1,1.0,0.0,0.9937106918238994,1,,1.0,
"(1669, 4615)",1.0,True,1,1,1.0,0.0,1.0,1,,1.0,
"(1669, 4614)",1.0,True,2,1,1.0,0.0,1.0,1,,1.0,
"(1669, 4613)",1.0,True,2,1,1.0,0.0,0.9739442946990117,1,,1.0,
"(1670, 4614)",1.0,True,2,1,1.0,0.0,0.9613656783468104,1,,1.0,
"(1687, 4633)",1.0,True,0,0,0.0,,,0
